In [ ]:
import pandas as pd
import time
import ingestion  
import config      

def chunk_list(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

city_batches = list(chunk_list(config.CITIES, 10))
final_parquet_name = "raw_for_day_2.parquet"

print(f"Total {len(config.CITIES)} cities divided into {len(city_batches)} batches.")

for idx, batch in enumerate(city_batches):
    print(f"\n--- Fetching Batch {idx+1}/{len(city_batches)} ---")
    
    batch_df = ingestion.fetch_all_cities(
        batch, 
        config.START_DATE, 
        config.END_DATE, 
        config.VARIABLES
    )
    
    if batch_df is not None and not batch_df.empty:
        if not os.path.exists(final_parquet_name):
            batch_df.to_parquet(final_parquet_name, index=False)
        else:
            existing_df = pd.read_parquet(final_parquet_name)
            combined_df = pd.concat([existing_df, batch_df], ignore_index=True)
            combined_df.to_parquet(final_parquet_name, index=False)
        
        print(f"Batch {idx+1} saved. Total rows in this batch: {len(batch_df)}")

    if idx < len(city_batches) - 1:
        print("Waiting 30 seconds between batches to avoid API blocking...")
        time.sleep(30)

print("\n🚀 ALL BATCHES PROCESSED AND SAVED TO PARQUET!")

In [ ]:
import requests
import pandas as pd
import time
import os
import config # Loading CITIES, VARIABLES, etc. from config.py

# Pasting functions directly here to avoid import errors
def fetch_forecast_final(city_name, lat, lon, variables):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, 
        "longitude": lon, 
        "daily": variables, 
        "timezone": "auto", 
        "forecast_days": 7
    }
    try:
        res = requests.get(url, params=params)
        if res.status_code == 200:
            df = pd.DataFrame(res.json()["daily"])
            df.insert(0, 'city', city_name)
            df.to_csv(f"{city_name.lower()}_forecast.csv", index=False)
            return True
    except: 
        return False
    return False

# Execution part
print("🚀 Fetching forecasts (fast process)...")
for city in config.CITIES:
    # Fetch if the forecast file doesn't exist (or refresh all)
    if not os.path.exists(f"{city['name'].lower()}_forecast.csv"):
        print(f"-> Saving forecast for: {city['name']}...")
        fetch_forecast_final(city['name'], city['lat'], city['lon'], config.VARIABLES)
        time.sleep(0.5) # Minimal delay for forecast requests

print("✅ All forecasts are ready!")

In [3]:
import pandas as pd
import glob
import os
import numpy as np
all_files = glob.glob("*_historical.csv")
df_list = []

for filename in all_files:
    temp_df = pd.read_csv(filename)
    temp_df['city'] = filename.replace('_historical.csv', '')
    df_list.append(temp_df)

master_df = pd.concat(df_list, ignore_index=True)
master_df['time'] = pd.to_datetime(master_df['time'])

total_rows = len(master_df)

gaps = {}
for city in master_df['city'].unique():
    city_df = master_df[master_df['city'] == city]
    actual_days = city_df['time'].nunique()
    expected_days = (city_df['time'].max() - city_df['time'].min()).days + 1
    if actual_days < expected_days:
        gaps[city] = expected_days - actual_days

null_counts = master_df.isnull().sum()
most_nulls = null_counts.sort_values(ascending=False)

actual_start = master_df['time'].min()
actual_end = master_df['time'].max()
requested_range = "2020-01-01 to 2026-04-19" 

print(f"📊 --- DATA AUDIT REPORT ---")
print(f"1. Total row count: {total_rows:,}")
print(f"2. Date gaps: Missing days found in {len(gaps)} cities.")
if gaps:
    print(f"    (Example: {list(gaps.items())[:3]})")

print(f"\n3. Top 5 Null Values:")
print(most_nulls.head(5))

print(f"\n4. Date Range:")
print(f"    Requested: {requested_range}")
print(f"    Actual: {actual_start.date()} to {actual_end.date()}")

📊 --- DATA AUDIT REPORT ---
1. Total row count: 215,928
2. Date gaps: Missing days found in 0 cities.

3. Top 5 Null Values:
city                   0
time                   0
temperature_2m_max     0
temperature_2m_min     0
temperature_2m_mean    0
dtype: int64

4. Date Range:
    Requested: 2020-01-01 to 2026-04-19
    Actual: 2020-01-01 to 2026-04-19
